# Import Library

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack
from collections import defaultdict
from difflib import get_close_matches
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix
from sentence_transformers import SentenceTransformer

# UI using PyQt5
import sys
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLineEdit, QPushButton, QListWidget, 
                             QLabel, QStackedWidget, QComboBox, QSpinBox, 
                             QDoubleSpinBox, QMessageBox, QListWidgetItem)
from PyQt5.QtCore import Qt, QEvent
from PyQt5.QtGui import QFont, QPixmap, QColor
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent
from PyQt5.QtCore import QUrl

# Content-Based Filtering

In [2]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.429,5,-7.826,1,0.0242,0.715000,0.000014,0.1200,0.173,146.365,4
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.490,4,-8.046,1,0.0457,0.016800,0.000534,0.1360,0.513,84.275,4
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.720,6,-7.776,1,0.1360,0.033700,0.001410,0.8260,0.832,138.556,4


### Selected only the relevant features

In [3]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo,spotify_preview_url,duration_ms
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,218920
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,238640
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,311014
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275,https://p.scdn.co/mp3-preview/90e41778392f27b6...,265506
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556,https://p.scdn.co/mp3-preview/7388c425022ced92...,223106


### Check data type

In [4]:
df_selected.dtypes

track_id                object
name                    object
artist                  object
tags                    object
genre                   object
year                     int64
danceability           float64
speechiness            float64
tempo                  float64
spotify_preview_url     object
duration_ms              int64
dtype: object

### Copy and clean

In [5]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [6]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [7]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [8]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [9]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [10]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name, tag, genre, year and index mapping

In [11]:
# lowercase song names in mapping
# Build the (song name) -> index map
name_to_index = {
    row['name'].lower().strip(): idx
    for idx, row in df_selected.iterrows()
}

In [12]:
# Create a dictionary to map (tag, genre, year) -> list of row indices
tag_genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using tag, genre, and year (all lowercase for consistency)
    key = (row['tags'].lower().strip(), row['genre'].lower().strip(), row['year'])
    
    # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    tag_genre_year_to_indices.setdefault(key, []).append(idx)

In [13]:
# Create a dictionary to map (genre, year) -> list of row indices
genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre and year (all lowercase for consistency)
    key = (row['genre'].lower().strip(), row['year'])
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_year_to_indices.setdefault(key, []).append(idx)

In [14]:
genre_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre (all lowercase for consistency)
    key = str(row['genre']).lower().strip()
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_to_indices.setdefault(key, []).append(idx)

### Recommendation function using name

##### convert time ms to min

In [15]:
def ms_to_minutes_seconds(ms):
    try:
        ms = int(ms)
        seconds = ms // 1000
        minutes = seconds // 60
        remaining_seconds = seconds % 60
        return f"{minutes}:{remaining_seconds:02d}"
    except Exception:
        return None

##### Calculation accuracy function

In [16]:
def calculate_accuracy_from_distance(distances):
    # Convert KNN distances into similarity percentages.
    return [(1 - d) * 100 for d in distances]

##### Validation for tempo, danceability and year

In [17]:
def validate_song_name(song_name):
    # Validate that the song exists in the dataset
    song_key = song_name.lower().strip()
    if song_key not in name_to_index:
        return None, f"Song '{song_name}' not found in the dataset."
    return song_key, None

##### Main Recommendation using name Function

In [18]:
def recommend_songs_by_name(song_name, n_recommendations=4):
    # Preprocess the input name (normalize: lowercase + remove spaces)
    song_key = song_name.lower().strip()
    
    # 1 - Check if the song exists in the mapping
    song_key, error = validate_song_name(song_name)
    if error:
        return error
    
    # 2 - Get the index of the input song directly from the dictionary
    idx = name_to_index[song_key] 
    
    # 3 - Find nearest neighbors using the KNN model
    distances, indices = knn_model.kneighbors(
        combined_matrix[idx], n_neighbors=n_recommendations + 1
    )
    
    # 4 - Exclude the first result (it’s always the song itself)
    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]
    
    # 5 - Get details of the input song
    input_song = df_selected.iloc[[idx]][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
    
    # 6 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
       
    
    # 7 - Add similarity score ("accuracy") column and calculation 
    input_song = input_song.assign(accuracy=["100%"])
    similarities = calculate_accuracy_from_distance(recommended_distances)
    recommendations = recommendations.assign(
        accuracy=[f"{s:.2f}%" for s in similarities]
    )
    
    # 8 - Merge input + recommendations into one DataFrame
    result = pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 9 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result

### Recommendation function using genre, tags, tempo and danceability

##### Calculation accuracy function

In [19]:
def calculate_accuracy(song_tempo, song_dance, input_tempo, input_dance):
    
    # Calculate similarity with input manually (tempo + danceability diff)
    tempo_diff = abs(song_tempo - input_tempo) / max(input_tempo, 1)
    dance_diff = abs(song_dance - input_dance)
    
    # Simple similarity formula: 1 - take 50% weight from tempo difference + 50% from danceability difference.
    sim = (1 - (0.5 * tempo_diff + 0.5 * dance_diff)) * 100
    
    # Clip the score between 0 and 100 to avoid negative or >100 values.
    sim = max(0, min(100, sim)) 
    
    return f"{sim:.2f}%"

##### Validation for tempo, danceability and year

In [20]:
def validate_inputs(song_tempo, danceability_value, year):
    
    # Validate tempo (must be numeric)
    try:
        song_tempo = float(song_tempo)
    except ValueError:
        return "Tempo must be a number (BPM)."

    # Validate danceability (0.0–1.0)
    try:
        danceability_value = float(danceability_value)
    except ValueError:
        return "Danceability must be between 0.0 and 1.0."
    if not (0.0 <= danceability_value <= 1.0):
        return "Danceability must be between 0.0 and 1.0."

    # Validate year (must be integer)
    try:
        year = int(year)
    except ValueError:
        return "Year must be an integer."

    return song_tempo, danceability_value, year


##### Main Recommendation using genre, tags, tempo and danceability Function

In [ ]:
def recommend_songs_by_genre_tempo_tags_danceability(
    song_genre, song_tempo, song_tags, danceability_value, year, n_recommendations=4
):
    # Normalize text fields (remove spaces + lowercase for consistency)
    song_genre = song_genre.lower().strip()
    song_tags = song_tags.lower().strip()

    # Run input validation
    validated = validate_inputs(song_tempo, danceability_value, year)
    
    # error message returned
    if isinstance(validated, str):  
        return validated 
    song_tempo, danceability_value, year = validated

    # 1 - fallback search
    candidates = None
    key_tag = (song_tags, song_genre, year)
    
    # First try - (tag + genre + year) exact match
    if key_tag in tag_genre_year_to_indices:
        candidates = df_selected.iloc[tag_genre_year_to_indices[key_tag]]

    # Second try - (genre + year)
    elif (song_genre, year) in genre_year_to_indices:
        candidates = df_selected.iloc[genre_year_to_indices[(song_genre, year)]]
        if song_tags:
            tagged = candidates[candidates['tags'].str.contains(song_tags, case=False, na=False)]
        if not tagged.empty:
            candidates = tagged

    # Third try - only genre
    elif song_genre in genre_to_indices:
        candidates = df_selected.iloc[genre_to_indices[song_genre]]
        if song_tags:
            tagged = candidates[candidates['tags'].str.contains(song_tags, case=False, na=False)]
        if not tagged.empty:
            candidates = tagged

    # If nothing matches - return not found message
    if candidates is None or candidates.empty:
        return f"No songs found in genre '{song_genre}' for year {year} (with/without tag '{song_tags}')."

    # 2 - Tempo filter ±5
    tempo_matches = candidates[(candidates['tempo'] >= song_tempo - 5) & (candidates['tempo'] <= song_tempo + 5)]
    if tempo_matches.empty:
       tempo_matches = candidates.copy()

    # 3 - Danceability filter ±0.1
    final_matches = tempo_matches[
        (tempo_matches['danceability'] >= danceability_value - 0.1) & 
        (tempo_matches['danceability'] <= danceability_value + 0.1)
    ].copy()
    if final_matches.empty:
        final_matches = tempo_matches.copy()

    # 4 - Pick the "closest" candidate (min absolute difference in danceability)
    final_matches['diff'] = (
        (final_matches['tempo'] - song_tempo).abs() +
        (final_matches['danceability'] - danceability_value).abs() + 
        (final_matches['year'] - year).abs()
    )
    idx = final_matches['diff'].idxmin()

    # 5 - Run KNN
    distances,indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]
    # recommended_distances = distances.flatten()[1:]

    # 6 - Get details of the input song
    input_song = df_selected.iloc[[idx]][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]
    
    # Check if input exactly matches the chosen candidate (strict match on 4 fields)
    is_exact_match = (
        input_song['genre'].iloc[0].lower().strip() == song_genre and
        input_song['tags'].iloc[0].lower().strip() == song_tags and
        input_song['year'].iloc[0] == year and
        abs(input_song['tempo'].iloc[0] - song_tempo) < 1e-6 and
        abs(input_song['danceability'].iloc[0] - danceability_value) < 1e-6
    )
    
    # 7 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]

    # 8 - Add similarity score ("accuracy") column and calculation 
    if is_exact_match:
        input_song = input_song.assign(accuracy=["100%"])
    else:
        ## Input song accuracy     
        acc = calculate_accuracy(
            input_song['tempo'].iloc[0],
            input_song['danceability'].iloc[0],
            song_tempo,
            danceability_value
        )
        input_song = input_song.assign(accuracy=[acc])
        
    ## Recommended songs accuracy
    # For each recommended song, calculate similarity to the user’s input values.
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']].copy()
    recommendations['accuracy_val'] = recommendations.apply(
        lambda row: float(calculate_accuracy(row['tempo'], row['danceability'], song_tempo, danceability_value).replace("%","")),
        axis=1
    )
    
    # Sort by accuracy high → low
    recommendations = recommendations.sort_values(by="accuracy_val", ascending=False)
    
    # Format accuracy
    recommendations['accuracy'] = recommendations['accuracy_val'].map(lambda s: f"{s:.2f}%")
    recommendations = recommendations.drop(columns="accuracy_val")

    # 9 - Merge input + recommendations into one DataFrame
    result =  pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 10 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result


# Collaborative Filtering System

## Then load the datasets 

In [22]:
user_history = pd.read_csv('data/newUserListeningHistory.csv')
music_info = pd.read_csv('data/newMusicInfo.csv')

## encode user and track id

In [23]:
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

## create sparse item matrix

In [24]:
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

## apply truncated SVD

In [25]:
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)

## convert index to id to map them

In [26]:
track_id_to_info = music_info.set_index('track_id')[['name', 'artist']].to_dict('index')
user_index_to_id = dict(enumerate(user_encoder.classes_))
track_index_to_id = dict(enumerate(track_encoder.classes_))

## recommend based on user ID then compare for similarity then suggest song that havent exist in the album of the user input id

In [27]:
def recommend_by_user_id(user_id_input, top_n=5):
    if user_id_input not in user_encoder.classes_:
        suggestions = get_close_matches(user_id_input, user_encoder.classes_, n=3)
        return f"User ID not found. Did you mean: {', '.join(suggestions)}?"

    user_index = user_encoder.transform([user_id_input])[0]
    target_vector = svd_matrix[user_index]
    
    # Calculate cosine similarity with ALL users
    similarities = cosine_similarity([target_vector], svd_matrix)[0]

    # Get top 5 similar users (excluding self)
    top_user_indices = np.argsort(-similarities)[1:6]
    
    print(f"\nInput User ID: {user_id_input}")
    print("Top 5 similar users and their similarity scores:")
    user_similarities = {}
    for i, user_idx in enumerate(top_user_indices, 1):
        sim_id = user_index_to_id[user_idx]
        sim_score = similarities[user_idx]
        user_similarities[user_idx] = sim_score
        print(f" {i}. {sim_id} (Similarity: {sim_score:.4f})")

    print(f"\nAnalyzing songs from these {len(top_user_indices)} similar users...")
    
    # Track which users listened to which songs
    track_to_user_similarities = defaultdict(list)
    
    for sim_idx in top_user_indices:
        uid = user_index_to_id[sim_idx]
        sim_score = similarities[sim_idx]
        user_tracks = user_history[user_history['user_index'] == sim_idx]['track_index'].unique()
        
        for tidx in user_tracks:
            track_to_user_similarities[tidx].append({
                'user_id': uid,
                'user_index': sim_idx,
                'similarity': sim_score
            })

    # Get songs user has already listened to
    listened = set(user_history[user_history['user_index'] == user_index]['track_index'].unique())
    
    # Score tracks based on weighted average of similar users' preferences
    track_scores = {}
    for track_idx, user_list in track_to_user_similarities.items():
        if track_idx not in listened:  # Only recommend unheard songs
            # Weight by similarity scores
            weighted_score = sum(user_info['similarity'] for user_info in user_list)
            track_scores[track_idx] = {
                'score': weighted_score,
                'contributing_users': user_list
            }
    
    # Sort by weighted score and get top N
    top_tracks = sorted(track_scores.items(), key=lambda x: x[1]['score'], reverse=True)[:top_n]
    
    recommendations = []
    for track_idx, track_data in top_tracks:
        track_id = track_index_to_id[track_idx]
        info = track_id_to_info.get(track_id, {'name': 'Unknown', 'artist': 'Unknown'})
        
        # Format contributing users with their similarities
        contributing_users = []
        for user_info in track_data['contributing_users']:
            contributing_users.append(f"{user_info['user_id']} ({user_info['similarity']:.4f})")

         # Get Spotify URL and duration from original data
        track_row = df[df['track_id'] == track_id]
        spotify_url = track_row['spotify_preview_url'].values[0] if not track_row.empty else 'Unknown'
        duration_ms = track_row['duration_ms'].values[0] if not track_row.empty else 0
        
        # Convert duration from ms to mm:ss format
        duration_formatted = f"{int(duration_ms)//60000}:{(int(duration_ms)%60000)//1000:02d}" if duration_ms else '0:00'
        
        recommendations.append({
            'track_id': track_id,
            'name': info['name'],
            'artist': info['artist'],
            'weighted_score': track_data['score'],
            'num_similar_users': len(track_data['contributing_users']),
            'contributing_users_with_similarity': '; '.join(contributing_users),  
            'spotify_preview_url': spotify_url,  
            'duration': duration_formatted  
        })
    
    return pd.DataFrame(recommendations)

# Hybrid Recommendation System

In [28]:
# Hybrid recommendation function (combines content-based and collaborative filtering)
def hybrid_recommendation(song_name, user_id, n_recommendations=5):
    """
    Hybrid recommendation that combines:
    1. Content-based filtering based on the searched song
    2. Collaborative filtering based on user preferences
    """
    # Get content-based recommendations
    content_recs = recommend_songs_by_name(song_name)
    
    # Get collaborative recommendations
    collab_recs = recommend_by_user_id(user_id)
    
    # Combine and deduplicate (simple hybrid approach)
    # In a real implementation, you would use a more sophisticated blending method
    hybrid_results = pd.concat([content_recs, collab_recs]).drop_duplicates(subset=['track_id'])

    print(hybrid_results[['name', 'weighted_score', 'accuracy']])
    
    # Return top n recommendations    
    return hybrid_results.head(n_recommendations)


# SBERT System

## 1. Load song list and SBERT Model

In [29]:
songs = pd.read_csv('data/newMusicInfo.csv')['name']
model = SentenceTransformer("all-MiniLM-L6-v2")  # lightweight, good for similarity

## 2. Encode all song into Embeddings

In [30]:
song_embeddings = model.encode(songs, convert_to_tensor=True)

## 3. Function to find closest match (returns DataFrame)

In [31]:
def sbert(user_input, top_k=5):
    # Encode user input
    input_embedding = model.encode([user_input], convert_to_tensor=True)

    # Compute cosine similarity
    similarities = cosine_similarity(
        input_embedding.cpu().numpy(), song_embeddings.cpu().numpy()
    )[0]

    # Get top K matches
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Build DataFrame
    df = pd.DataFrame({
        "Song": [songs[i] for i in top_indices],
        "Similarity": [similarities[i] for i in top_indices]
    })

    print(df)
    
    return df

# UI

## Custome widget for song item  with modern design

In [32]:
# Global variable to track currently playing song
current_playing_widget = None

# Custom widget for song item with modern design
class SongWidget(QWidget):
    def __init__(self, name, artist, preview_url, duration, accuracy=None, parent=None):
        super().__init__(parent)
        self.preview_url = preview_url
        self.is_playing = False
        self.media_player = QMediaPlayer()
        self.media_player.stateChanged.connect(self.on_player_state_changed)  # Connect state change
        
        # Set up the widget with modern styling
        self.setFixedHeight(80)
        self.setStyleSheet("""
            SongWidget {
                background-color: white;
                border-radius: 12px;
                margin: 4px;
                border: 0px;
            }
            SongWidget:hover {
                background-color: #f8f9fa;
                border: 1px solid #e9ecef;
            }
        """)
        
        layout = QHBoxLayout()
        layout.setContentsMargins(16, 12, 16, 12)
        layout.setSpacing(16)
        
        # Modern album art placeholder
        disc_label = QLabel()
        disc_label.setFixedSize(56, 56)
        disc_label.setStyleSheet("""
            background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                stop:0 #667eea, stop:1 #764ba2);
            border-radius: 28px;
            color: white;
            font-size: 24px;
            font-weight: bold;
        """)
        disc_label.setAlignment(Qt.AlignCenter)
        disc_label.setText("♪")
        layout.addWidget(disc_label)
        
        # Song information with better typography
        info_layout = QVBoxLayout()
        info_layout.setSpacing(4)
        
        name_label = QLabel(name)
        name_label.setStyleSheet("""
            font-size: 16px;
            font-weight: 600;
            color: #1a1a1a;
            margin: 0;
            border: 0px;
        """)
        name_label.setWordWrap(True)
        
        artist_label = QLabel(artist)
        artist_label.setStyleSheet("""
            font-size: 14px;
            color: #6c757d;
            margin: 0;
            border: 0px;
        """)
        artist_label.setWordWrap(True)
        
        info_layout.addWidget(name_label)
        info_layout.addWidget(artist_label)
        info_layout.addStretch()
        
        layout.addLayout(info_layout)
        layout.addStretch()
        
        # Accuracy badge if provided
        if accuracy:
            accuracy_label = QLabel(accuracy)
            accuracy_label.setStyleSheet("""
                background-color: #28a745;
                color: white;
                padding: 4px 8px;
                border-radius: 12px;
                font-size: 12px;
                font-weight: 600;
                border: 0px;
            """)
            accuracy_label.setFixedHeight(24)
            layout.addWidget(accuracy_label)
        
        # Duration
        duration_label = QLabel(duration)
        duration_label.setStyleSheet("""
            font-size: 14px;
            color: #6c757d;
            margin-left: 12px;
            border: 0px;
        """)
        layout.addWidget(duration_label)
        
        # Modern play button
        self.play_btn = QPushButton()
        self.play_btn.setFixedSize(44, 44)
        self.play_btn.setFocusPolicy(Qt.NoFocus)  # Disable focus to prevent blue highlight
        self.play_btn.setStyleSheet("""
            QPushButton {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #ff6b6b, stop:1 #ee5a24);
                border: none;
                border-radius: 22px;
                color: white;
                font-size: 16px;
                font-weight: bold;
                outline: none;
                selection-background-color: transparent;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #ff5252, stop:1 #ff3838);
            }
            QPushButton:pressed {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #ff3838, stop:1 #ff1744);
            }
            QPushButton:focus {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #ff6b6b, stop:1 #ee5a24);
                outline: none;
                border: none;
            }
            QPushButton:checked {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #ff6b6b, stop:1 #ee5a24);
            }
        """)
        self.play_btn.setText("▶")
        self.play_btn.clicked.connect(self.toggle_play)
        layout.addWidget(self.play_btn)
        
        self.setLayout(layout)
    
    def on_player_state_changed(self, state):
        # Handle player state changes safely
        if state == QMediaPlayer.StoppedState and self.is_playing:
            # Only update UI if widget still exists
            try:
                self.play_btn.setText("▶")
                self.is_playing = False
            except RuntimeError:
                # Widget was deleted, do nothing
                pass
    
    def toggle_play(self):
        global current_playing_widget
        
        # Stop currently playing song if any
        if current_playing_widget and current_playing_widget != self:
            try:
                current_playing_widget.stop_playback()
            except RuntimeError:
                # Widget was deleted, just reset the global variable
                current_playing_widget = None
        
        if self.is_playing:
            self.stop_playback()
        else:
            if self.preview_url and self.preview_url != 'Unknown':
                try:
                    self.media_player.setMedia(QMediaContent(QUrl(self.preview_url)))
                    self.media_player.play()
                    self.play_btn.setText("⏸")
                    self.is_playing = True
                    current_playing_widget = self
                except RuntimeError:
                    # Widget was deleted, do nothing
                    pass
    
    def stop_playback(self):
        try:
            self.media_player.stop()
            self.play_btn.setText("▶")
            self.is_playing = False
        except RuntimeError:
            # Widget was deleted, do nothing
            pass
    
    def cleanup(self):
        # Clean up resources when widget is about to be deleted
        try:
            self.media_player.stop()
            self.media_player.deleteLater()
        except:
            pass

## User Login Page

In [33]:
# Modern Login/User Profile Widget
class UserProfileWidget(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        layout = QHBoxLayout()
        layout.setContentsMargins(0, 0, 0, 0)
        
        # User icon
        self.user_icon = QLabel("👤")
        self.user_icon.setStyleSheet("""
            font-size: 20px;
            padding: 8px;
            background-color: #f8f9fa;
            border-radius: 20px;
            min-width: 20px;
        """)
        layout.addWidget(self.user_icon)
        
        # User info or login prompt
        self.user_label = QLabel("Not logged in")
        self.user_label.setStyleSheet("""
            font-size: 14px;
            color: #6c757d;
            padding-right: 8px;
        """)
        layout.addWidget(self.user_label)
                
        # Login/Logout button
        self.auth_btn = QPushButton("Login")
        self.auth_btn.setStyleSheet("""
            QPushButton {
                background-color: #007bff;
                color: white;
                border: none;
                padding: 8px 16px;
                border-radius: 6px;
                font-weight: 600;
                font-size: 12px;
            }
            QPushButton:hover {
                background-color: #0056b3;
            }
        """)
        self.auth_btn.clicked.connect(self.handle_auth)
        layout.addWidget(self.auth_btn)
        
        self.setLayout(layout)
    
    def handle_auth(self):
        if self.parent.parent.current_user:
            # Logout
            self.parent.parent.current_user = None
            self.update_ui()
            self.parent.parent.update_recommendations()
            QMessageBox.information(self, "Success", "Logged out successfully!")
        else:
            # Go to login page
            self.parent.parent.stacked_widget.setCurrentIndex(1)
    
    def update_ui(self):
        if self.parent.parent.current_user:
            self.user_label.setText(f"Welcome, {self.parent.parent.current_user}")
            self.user_label.setStyleSheet("""
                font-size: 14px;
                color: #28a745;
                font-weight: 600;
                padding-left: 8px;
            """)
            self.auth_btn.setText("Logout")
            self.auth_btn.setStyleSheet("""
                QPushButton {
                    background-color: #dc3545;
                    color: white;
                    border: none;
                    padding: 8px 16px;
                    border-radius: 6px;
                    font-weight: 600;
                    font-size: 12px;
                }
                QPushButton:hover {
                    background-color: #c82333;
                }
            """)
        else:
            self.user_label.setText("Not logged in")
            self.user_label.setStyleSheet("""
                font-size: 14px;
                color: #6c757d;
                right: 50px;
            """)
            self.auth_btn.setText("Login")
            self.auth_btn.setStyleSheet("""
                QPushButton {
                    background-color: #007bff;
                    color: white;
                    border: none;
                    padding: 8px 16px;
                    border-radius: 6px;
                    font-weight: 600;
                    font-size: 12px;
                }
                QPushButton:hover {
                    background-color: #0056b3;
                }
            """)

# Modern Login Page
class LoginPage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        layout = QVBoxLayout()
        layout.setContentsMargins(50, 50, 50, 50)
        layout.setSpacing(30)
        
        # Back button
        back_btn = QPushButton("< Back")
        back_btn.setStyleSheet("""
            QPushButton {
                background: none;
                border: none;
                color: #007bff;
                font-size: 16px;
                font-weight: 600;
                text-align: left;
                padding: 8px 0px;
            }
            QPushButton:hover {
                color: #0056b3;
                text-decoration: underline;
            }
        """)
        back_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(0))
        layout.addWidget(back_btn)
        
        layout.addStretch()
        
        # Login card
        card_widget = QWidget()
        card_widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        card_widget.setMaximumWidth(400)
        card_widget.setMinimumHeight(300)
        
        card_layout = QVBoxLayout()
        card_layout.setContentsMargins(40, 40, 40, 40)
        card_layout.setSpacing(25)
        
        # Title
        title = QLabel("Welcome Back")
        title.setStyleSheet("""
            font-size: 28px;
            font-weight: 700;
            color: #1a1a1a;
            text-align: center;
            border: 0px;
        """)
        title.setAlignment(Qt.AlignCenter)
        card_layout.addWidget(title)
        
        subtitle = QLabel("Sign in to get personalized recommendations")
        subtitle.setStyleSheet("""
            font-size: 16px;
            color: #6c757d;
            text-align: center;
            border: 0px;
        """)
        subtitle.setAlignment(Qt.AlignCenter)
        card_layout.addWidget(subtitle)
        
        # User ID input
        self.user_id_input = QLineEdit()
        self.user_id_input.setPlaceholderText("Enter your User ID")
        self.user_id_input.setStyleSheet("""
            QLineEdit {
                padding: 16px;
                border: 2px solid #e9ecef;
                border-radius: 12px;
                font-size: 16px;
                background-color: #f8f9fa;
            }
            QLineEdit:focus {
                border-color: #007bff;
                background-color: white;
            }
        """)
        self.user_id_input.returnPressed.connect(self.login)
        card_layout.addWidget(self.user_id_input)
        
        # Login button
        login_btn = QPushButton("Sign In")
        login_btn.setStyleSheet("""
            QPushButton {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #007bff, stop:1 #0056b3);
                color: white;
                border: none;
                padding: 16px;
                border-radius: 12px;
                font-size: 16px;
                font-weight: 600;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #0056b3, stop:1 #004085);
            }
        """)
        login_btn.clicked.connect(self.login)
        card_layout.addWidget(login_btn)
        
        card_widget.setLayout(card_layout)
        
        # Center the card
        card_container = QHBoxLayout()
        card_container.addStretch()
        card_container.addWidget(card_widget)
        card_container.addStretch()
        
        layout.addLayout(card_container)
        layout.addStretch()
        
        self.setLayout(layout)
    
    def login(self):
        user_id = self.user_id_input.text().strip()
        if user_id:
            self.parent.current_user = user_id
            self.parent.home_page.user_profile.update_ui()
            self.parent.update_recommendations()
            self.parent.stacked_widget.setCurrentIndex(0)
            QMessageBox.information(self, "Success", f"Welcome back, {user_id}!")
            self.user_id_input.clear()
        else:
            QMessageBox.warning(self, "Error", "Please enter a valid User ID")


## Filter Page

In [34]:
# Modern Filter Page
class FilterPage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        layout = QVBoxLayout()
        layout.setContentsMargins(20, 20, 20, 20)
        layout.setSpacing(20)
        
        # Header
        header_layout = QHBoxLayout()
        
        back_btn = QPushButton("< Back")
        back_btn.setStyleSheet("""
            QPushButton {
                background: none;
                border: none;
                color: #007bff;
                font-size: 16px;
                font-weight: 600;
                text-align: left;
                padding: 8px 0px;
            }
            QPushButton:hover {
                color: #0056b3;
            }
        """)
        back_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(0))
        header_layout.addWidget(back_btn)
        
        header_layout.addStretch()
        
        title = QLabel("Advanced Search")
        title.setStyleSheet("""
            font-size: 24px;
            font-weight: 700;
            color: #1a1a1a;
        """)
        header_layout.addWidget(title)
        header_layout.addStretch()
        
        layout.addLayout(header_layout)
        
        # Main content
        content_layout = QHBoxLayout()
        
        # Filters panel
        filters_widget = QWidget()
        filters_widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        filters_widget.setMaximumWidth(350)
        
        filters_layout = QVBoxLayout()
        filters_layout.setContentsMargins(30, 30, 30, 30)
        filters_layout.setSpacing(20)
        
        # Filter controls with modern styling
        filter_style = """
            QLabel {
                font-size: 14px;
                font-weight: 600;
                color: #495057;
                margin-bottom: 8px;
                border: 0px;
            }
            QComboBox, QSpinBox, QDoubleSpinBox {
                padding: 12px;
                border: 2px solid #ced4da;
                border-radius: 8px;
                font-size: 14px;
                background-color: white;
            }
            QComboBox:focus, QSpinBox:focus, QDoubleSpinBox:focus {
                border-color: #007bff;
                border: 2px solid #007bff;
            }
        """
        
        # Genre
        genre_label = QLabel("Genre")
        genre_label.setStyleSheet(filter_style)
        filters_layout.addWidget(genre_label)
        
        self.genre_combo = QComboBox()
        self.genre_combo.setStyleSheet(filter_style)
        genres = ['RNB', 'ROCK', 'POP', 'METAL', 'ELECTRONIC', 'JAZZ', 'PUNK', 
                 'COUNTRY', 'FOLK', 'REGGAE', 'RAP', 'BLUES', 'NEW AGE', 'LATIN', 'WORLD']
        self.genre_combo.addItems(genres)
        filters_layout.addWidget(self.genre_combo)
        
        # Tags
        tags_label = QLabel("Tags")
        tags_label.setStyleSheet(filter_style)
        filters_layout.addWidget(tags_label)
        
        self.tags_combo = QComboBox()
        self.tags_combo.setStyleSheet(filter_style)
        tags = ['PROGRESSIVE_METAL', 'EMO', 'DANCE', 'PSYCHEDELIC', 'GOTHIC_METAL', 
                'DOWNTEMPO', 'POWER_METAL', 'SYNTHPOP', 'DEATH_METAL', 'PROGRESSIVE_ROCK', 
                'THRASH_METAL', 'SINGER_SONGWRITER', 'CLASSICAL', 'PIANO', 'CLASSIC_ROCK', 
                'INDIE_ROCK', 'INDIE', 'SOUNDTRACK', 'LOUNGE', 'BLACK_METAL', 'AVANT_GARDE', 
                'J_POP', 'HEAVY_METAL', 'POST_ROCK', 'SKA', 'HARDCORE', 'GUITAR', 'HIP_HOP', 
                'SYMPHONIC_METAL', 'NU_METAL', 'HARD_ROCK', 'POST_PUNK', 'INDUSTRIAL', 
                'GRUNGE', 'AMBIENT', 'TRIP_HOP', 'BRITISH', 'JAPANESE', 'METALCORE', 
                'INSTRUMENTAL', 'NEW_WAVE', 'FRENCH', 'AMERICAN', 'TECHNO', 'GOTHIC', 
                'ROCK', 'NOISE', 'BLUES', 'ELECTRO', 'SCREAMO', 'POP_ROCK', 'HOUSE', 
                'DOOM_METAL', 'SOUL', 'CHILL', 'BRITPOP', 'BLUES_ROCK', 'RUSSIAN', 
                'DRUM_AND_BASS', 'GRINDCORE', 'ELECTRONIC', 'POST_HARDCORE', 
                'PSYCHEDELIC_ROCK', 'RNB', 'POLISH', 'COVER', 'PUNK_ROCK', 'LOVE', 
                'FEMALE_VOCALISTS', 'EXPERIMENTAL', 'ALTERNATIVE_ROCK', 'MALE_VOCALISTS', 
                'OLDIES', 'CHILLOUT', 'ACOUSTIC', 'MELODIC_DEATH_METAL', 'SWEDISH', 
                'MELLOW', 'ALTERNATIVE', 'DARK_AMBIENT', 'GERMAN', 'TRANCE', 'IDM', 'INDIE_POP']
        
        sorted_tags = sorted(tags)
        self.tags_combo.addItem('BEAUTIFUL')
        for tag in sorted_tags:
            self.tags_combo.addItem(tag)
        filters_layout.addWidget(self.tags_combo)
        
        # Year
        year_label = QLabel("Year")
        year_label.setStyleSheet(filter_style)
        filters_layout.addWidget(year_label)
        
        self.year_input = QSpinBox()
        self.year_input.setStyleSheet(filter_style)
        self.year_input.setRange(1900, 2030)
        self.year_input.setValue(2020)
        filters_layout.addWidget(self.year_input)
        
        # Tempo
        tempo_label = QLabel("Tempo (BPM)")
        tempo_label.setStyleSheet(filter_style)
        filters_layout.addWidget(tempo_label)
        
        self.tempo_input = QSpinBox()
        self.tempo_input.setStyleSheet(filter_style)
        self.tempo_input.setRange(0, 239)
        self.tempo_input.setValue(120)
        filters_layout.addWidget(self.tempo_input)
        
        # Danceability
        dance_label = QLabel("Danceability")
        dance_label.setStyleSheet(filter_style)
        filters_layout.addWidget(dance_label)
        
        self.dance_input = QDoubleSpinBox()
        self.dance_input.setStyleSheet(filter_style)
        self.dance_input.setRange(0.0, 1.0)
        self.dance_input.setSingleStep(0.1)
        self.dance_input.setValue(0.5)
        filters_layout.addWidget(self.dance_input)
        
        # Apply button
        filters_layout.addSpacing(10)
        apply_btn = QPushButton("Apply Filters")
        apply_btn.setStyleSheet("""
            QPushButton {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #28a745, stop:1 #1e7e34);
                color: white;
                border: none;
                padding: 16px;
                border-radius: 12px;
                font-size: 16px;
                font-weight: 600;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #218838, stop:1 #1c7430);
            }
        """)
        apply_btn.clicked.connect(self.apply_filters)
        filters_layout.addWidget(apply_btn)
        
        filters_layout.addStretch()
        filters_widget.setLayout(filters_layout)
        
        # Results panel
        results_widget = QWidget()
        results_widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
                border: 0px;
            }
        """)
        
        results_layout = QVBoxLayout()
        results_layout.setContentsMargins(20, 20, 20, 20)
        
        results_title = QLabel("Results")
        results_title.setStyleSheet("""
            font-size: 18px;
            font-weight: 600;
            color: #1a1a1a;
            margin-bottom: 16px;
            border: 0px;
        """)
        results_layout.addWidget(results_title)
        
        self.results_list = QListWidget()
        self.results_list.setStyleSheet("""
            QListWidget {
                border: none;
                background-color: transparent;
            }
            QListWidget::item {
                border: none;
                margin: 4px 0px;
            }
        """)
        results_layout.addWidget(self.results_list)
        
        results_widget.setLayout(results_layout)
        
        # Add panels to content layout
        content_layout.addWidget(filters_widget)
        content_layout.addWidget(results_widget, 1)
        
        layout.addLayout(content_layout)
        self.setLayout(layout)
    
    def apply_filters(self):
        selected_tag = self.tags_combo.currentText()

        # Clean up any existing song widgets
        for i in range(self.results_list.count()):
            item = self.results_list.item(i)
            widget = self.results_list.itemWidget(item)
            if widget and hasattr(widget, 'cleanup'):
                try:
                    widget.cleanup()
                except:
                    pass
        
        recommendations = recommend_songs_by_genre_tempo_tags_danceability(
            song_genre=self.genre_combo.currentText(),
            song_tempo=str(self.tempo_input.value()),
            song_tags=selected_tag,
            danceability_value=str(self.dance_input.value()),
            year=str(self.year_input.value())
        )
        
        self.results_list.clear()
        
        if isinstance(recommendations, str):
            # Error message
            error_item = QListWidgetItem()
            error_widget = QLabel(recommendations)
            error_widget.setStyleSheet("""
                padding: 20px;
                color: #dc3545;
                font-size: 16px;
                text-align: center;
                border: 0px;
            """)
            error_widget.setAlignment(Qt.AlignCenter)
            error_widget.setWordWrap(True)
            error_item.setSizeHint(error_widget.sizeHint())
            self.results_list.addItem(error_item)
            self.results_list.setItemWidget(error_item, error_widget)
        else:
            for _, row in recommendations.iterrows():
                list_item = QListWidgetItem()
                song_widget = SongWidget(
                    row['name'], 
                    row['artist'], 
                    row['spotify_preview_url'],
                    row['duration'],
                    row.get('accuracy', None)
                )
                list_item.setSizeHint(song_widget.sizeHint())
                self.results_list.addItem(list_item)
                self.results_list.setItemWidget(list_item, song_widget)


## Home page

In [35]:
# Modern Home Page
class HomePage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.suggestion_list = None
        self.initUI()
        
    def initUI(self):
        layout = QVBoxLayout()
        layout.setContentsMargins(20, 20, 20, 20)
        layout.setSpacing(20)
        
        # Header with user profile
        header_layout = QVBoxLayout()
        header_layout.setSpacing(20)
        
        # Top bar with user info
        top_bar = QHBoxLayout()
        
        app_title = QLabel(" Music Discovery ")
        app_title.setStyleSheet("""
            font-size: 32px;
            font-weight: 800;
            color: #1a1a1a;
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            border-radius: 20px;
        """)
        top_bar.addWidget(app_title)
        
        top_bar.addStretch()
        
        # User profile widget
        self.user_profile = UserProfileWidget(self)
        top_bar.addWidget(self.user_profile)
        
        header_layout.addLayout(top_bar)
        
        # Search section
        search_card = QWidget()
        search_card.setStyleSheet("""
            QWidget {
                background-color: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        
        search_layout = QVBoxLayout()
        search_layout.setContentsMargins(30, 30, 30, 30)
        search_layout.setSpacing(20)
        
        search_title = QLabel("Find Your Next Favorite Song")
        search_title.setStyleSheet("""
            font-size: 20px;
            font-weight: 600;
            color: #1a1a1a;
            text-align: center;
            border: 0px;
        """)
        search_title.setAlignment(Qt.AlignCenter)
        search_layout.addWidget(search_title)
        
        # Search bar container
        search_container = QVBoxLayout()
        search_container.setSpacing(0)
        
        # Search input row
        search_input_row = QHBoxLayout()
        search_input_row.setSpacing(12)
        
        self.search_input = QLineEdit()
        self.search_input.setPlaceholderText("Search for a song to get recommendations...")
        self.search_input.setStyleSheet("""
            QLineEdit {
                padding: 16px 20px;
                border: 2px solid #e9ecef;
                border-radius: 25px;
                font-size: 16px;
                background-color: #f8f9fa;
            }
            QLineEdit:focus {
                border-color: #007bff;
                background-color: white;
            }
        """)
        self.search_input.returnPressed.connect(self.search_songs)
        self.search_input.textChanged.connect(self.on_text_changed)
        self.search_input.installEventFilter(self)
        search_input_row.addWidget(self.search_input)
        
        search_btn = QPushButton("Search")
        search_btn.setStyleSheet("""
            QPushButton {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #007bff, stop:1 #0056b3);
                color: white;
                border: none;
                padding: 16px 24px;
                border-radius: 25px;
                font-size: 16px;
                font-weight: 600;
                min-width: 100px;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #0056b3, stop:1 #004085);
            }
        """)
        search_btn.clicked.connect(self.search_songs)
        search_input_row.addWidget(search_btn)
        
        advanced_btn = QPushButton("Advanced")
        advanced_btn.setStyleSheet("""
            QPushButton {
                background-color: transparent;
                color: #6c757d;
                border: 2px solid #e9ecef;
                padding: 16px 24px;
                border-radius: 25px;
                font-size: 16px;
                font-weight: 600;
                min-width: 100px;
            }
            QPushButton:hover {
                background-color: #f8f9fa;
                border-color: #007bff;
                color: #007bff;
            }
        """)
        advanced_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(2))
        search_input_row.addWidget(advanced_btn)
        
        search_container.addLayout(search_input_row)
        
        # Suggestion list (initially hidden)
        self.suggestion_list = QListWidget()
        self.suggestion_list.setStyleSheet("""
            QListWidget {
                background-color: white;
                border: 1px solid #e9ecef;
                border-top: none;
                border-radius: 0px 0px 8px 8px;
                padding: 4px;
                font-size: 14px;
                max-height: 150px;
            }
            QListWidget::item {
                padding: 8px 12px;
                border-radius: 4px;
            }
            QListWidget::item:selected {
                background-color: #e9ecef;
            }
            QListWidget::item:hover {
                background-color: #f8f9fa;
            }
        """)
        self.suggestion_list.hide()
        self.suggestion_list.installEventFilter(self)
        self.suggestion_list.itemClicked.connect(self.on_suggestion_clicked)
        search_container.addWidget(self.suggestion_list)
        
        search_layout.addLayout(search_container)
        search_card.setLayout(search_layout)
        header_layout.addWidget(search_card)
        
        layout.addLayout(header_layout)
        
        # Content area with tabs
        content_widget = QWidget()
        content_layout = QHBoxLayout()
        content_layout.setContentsMargins(0, 0, 0, 0)
        content_layout.setSpacing(20)
        
        # Search results column
        search_column = QWidget()
        search_column.setStyleSheet("""
            QWidget {
                background-color: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        
        search_column_layout = QVBoxLayout()
        search_column_layout.setContentsMargins(20, 20, 20, 20)
        search_column_layout.setSpacing(16)
        
        search_results_title = QLabel("Search Results")
        search_results_title.setStyleSheet("""
            font-size: 18px;
            font-weight: 600;
            color: #1a1a1a;
            border: 0px;
        """)
        search_column_layout.addWidget(search_results_title)
        
        self.search_results_list = QListWidget()
        self.search_results_list.setStyleSheet("""
            QListWidget {
                border: none;
                background-color: transparent;
            }
            QListWidget::item {
                border: none;
                margin: 4px 0px;
            }
        """)
        search_column_layout.addWidget(self.search_results_list)
        
        search_column.setLayout(search_column_layout)
        content_layout.addWidget(search_column)
        
        # Recommendations column
        recommendations_column = QWidget()
        recommendations_column.setStyleSheet("""
            QWidget {
                background-color: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        
        rec_column_layout = QVBoxLayout()
        rec_column_layout.setContentsMargins(20, 20, 20, 20)
        rec_column_layout.setSpacing(16)
        
        # Hybrid recommendations section
        self.hybrid_label = QLabel("Hybrid Recommendations")
        self.hybrid_label.setStyleSheet("""
            font-size: 18px;
            font-weight: 600;
            color: #1a1a1a;
            border: 0px;
        """)
        self.hybrid_label.hide()
        rec_column_layout.addWidget(self.hybrid_label)
        
        self.hybrid_list = QListWidget()
        self.hybrid_list.setStyleSheet("""
            QListWidget {
                border: none;
                background-color: transparent;
            }
            QListWidget::item {
                border: none;
                margin: 4px 0px;
            }
        """)
        self.hybrid_list.hide()
        rec_column_layout.addWidget(self.hybrid_list)
        
        # User recommendations section
        self.user_rec_label = QLabel("Recommended for You")
        self.user_rec_label.setStyleSheet("""
            font-size: 18px;
            font-weight: 600;
            color: #1a1a1a;
            border: 0px;
        """)
        rec_column_layout.addWidget(self.user_rec_label)
        
        self.user_rec_list = QListWidget()
        self.user_rec_list.setStyleSheet("""
            QListWidget {
                border: none;
                background-color: transparent;
            }
            QListWidget::item {
                border: none;
                margin: 4px 0px;
            }
        """)
        rec_column_layout.addWidget(self.user_rec_list)
        
        # Show login prompt if not logged in
        if not self.parent.current_user:
            login_prompt = QWidget()
            login_prompt.setStyleSheet("""
                QWidget {
                    background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                        stop:0 #f8f9fa, stop:1 #e9ecef);
                    border-radius: 12px;
                    margin: 8px;
                }
            """)
            
            prompt_layout = QVBoxLayout()
            prompt_layout.setContentsMargins(20, 20, 20, 20)
            prompt_layout.setSpacing(12)
            
            prompt_title = QLabel("Get Personalized Recommendations")
            prompt_title.setStyleSheet("""
                font-size: 16px;
                font-weight: 600;
                color: #495057;
                text-align: center;
                border: 0px;
            """)
            prompt_title.setAlignment(Qt.AlignCenter)
            prompt_layout.addWidget(prompt_title)
            
            prompt_text = QLabel("Login to discover music based on your listening history and preferences.")
            prompt_text.setStyleSheet("""
                font-size: 14px;
                color: #6c757d;
                text-align: center;
                border: 0px;
            """)
            prompt_text.setAlignment(Qt.AlignCenter)
            prompt_text.setWordWrap(True)
            prompt_layout.addWidget(prompt_text)
            
            prompt_btn = QPushButton("Login Now")
            prompt_btn.setStyleSheet("""
                QPushButton {
                    background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                        stop:0 #007bff, stop:1 #0056b3);RNB
                    color: white;
                    border: none;
                    padding: 12px 24px;
                    border-radius: 8px;
                    font-size: 14px;
                    font-weight: 600;
                }
                QPushButton:hover {
                    background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                        stop:0 #0056b3, stop:1 #004085);
                }
            """)
            prompt_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(1))
            prompt_layout.addWidget(prompt_btn, alignment=Qt.AlignCenter)
            
            login_prompt.setLayout(prompt_layout)
            
            list_item = QListWidgetItem()
            list_item.setSizeHint(login_prompt.sizeHint())
            self.user_rec_list.addItem(list_item)
            self.user_rec_list.setItemWidget(list_item, login_prompt)
        
        recommendations_column.setLayout(rec_column_layout)
        content_layout.addWidget(recommendations_column)
        
        content_widget.setLayout(content_layout)
        layout.addWidget(content_widget)
        
        self.setLayout(layout)
    
    def eventFilter(self, obj, event):
        # Handle focus events
        if event.type() == QEvent.FocusOut:
            if obj == self.search_input:
                # Hide suggestions after a short delay to allow for selection
                QApplication.processEvents()
                self.hide_suggestions()
        
        # Handle key events for the suggestion list
        if obj == self.suggestion_list and event.type() == QEvent.KeyPress:
            if event.key() in [Qt.Key_Return, Qt.Key_Enter]:
                current_item = self.suggestion_list.currentItem()
                if current_item:
                    self.on_suggestion_clicked(current_item)
                    return True
            elif event.key() == Qt.Key_Escape:
                self.hide_suggestions()
                self.search_input.setFocus()
                return True
        
        return super().eventFilter(obj, event)
    
    def on_text_changed(self, text):
        if text.strip():
            # Get suggestions using SBERT
            suggestions = sbert(text, top_k=5)
            
            # Update suggestion list
            self.suggestion_list.clear()
            
            for _, row in suggestions.iterrows():
                item = QListWidgetItem(row['Song'])
                item.setData(Qt.UserRole, row['Song'])  # Store the full song name
                self.suggestion_list.addItem(item)
            
            # Show the suggestion list if there are suggestions
            if self.suggestion_list.count() > 0:
                self.suggestion_list.show()
            else:
                self.suggestion_list.hide()
        else:
            self.suggestion_list.hide()
    
    def on_suggestion_clicked(self, item):
        # Set the search input to the selected suggestion
        self.search_input.setText(item.data(Qt.UserRole))
        self.suggestion_list.hide()
        self.search_input.setFocus()
    
    def search_songs(self):
        # Hide suggestions when searching
        self.suggestion_list.hide

        # Clean up any existing song widgets
        for i in range(self.search_results_list.count()):
            item = self.search_results_list.item(i)
            widget = self.search_results_list.itemWidget(item)
            if widget and hasattr(widget, 'cleanup'):
                try:
                    widget.cleanup()
                except:
                    pass
        
        song_name = self.search_input.text().strip()
        if song_name:
            # Get content-based recommendations
            recommendations = recommend_songs_by_name(song_name)
            self.search_results_list.clear()
            
            if isinstance(recommendations, str):
                # Error message
                error_item = QListWidgetItem()
                error_widget = QLabel(recommendations)
                error_widget.setStyleSheet("""
                    padding: 20px;
                    color: #dc3545;
                    font-size: 16px;
                    text-align: center;
                    background-color: #f8d7da;
                    border-radius: 8px;
                    margin: 8px;
                    border: 0px;
                """)
                error_widget.setAlignment(Qt.AlignCenter)
                error_widget.setWordWrap(True)
                error_item.setSizeHint(error_widget.sizeHint())
                self.search_results_list.addItem(error_item)
                self.search_results_list.setItemWidget(error_item, error_widget)
            else:
                for _, row in recommendations.iterrows():
                    list_item = QListWidgetItem()
                    song_widget = SongWidget(
                        row['name'], 
                        row['artist'], 
                        row['spotify_preview_url'],
                        row['duration'],
                        row.get('accuracy', None)
                    )
                    list_item.setSizeHint(song_widget.sizeHint())
                    self.search_results_list.addItem(list_item)
                    self.search_results_list.setItemWidget(list_item, song_widget)
            
            # Show hybrid recommendations if user is logged in
            if self.parent.current_user:
                try:
                    hybrid_recs = hybrid_recommendation(song_name, self.parent.current_user, 10)
                    self.hybrid_list.clear()
                    
                    if isinstance(hybrid_recs, str):
                        # Error in collaborative filtering
                        error_item = QListWidgetItem()
                        error_widget = QLabel("Unable to generate hybrid recommendations")
                        error_widget.setStyleSheet("""
                            padding: 16px;
                            color: #856404;
                            font-size: 14px;
                            text-align: center;
                            background-color: #fff3cd;
                            border-radius: 8px;
                            margin: 8px;
                            border: 0px;
                        """)
                        error_widget.setAlignment(Qt.AlignCenter)
                        error_item.setSizeHint(error_widget.sizeHint())
                        self.hybrid_list.addItem(error_item)
                        self.hybrid_list.setItemWidget(error_item, error_widget)
                    else:
                        for _, row in hybrid_recs.iterrows():
                            list_item = QListWidgetItem()
                            song_widget = SongWidget(
                                row['name'], 
                                row['artist'], 
                                row['spotify_preview_url'],
                                row['duration']
                            )
                            list_item.setSizeHint(song_widget.sizeHint())
                            self.hybrid_list.addItem(list_item)
                            self.hybrid_list.setItemWidget(list_item, song_widget)
                    
                    # Show hybrid section
                    self.hybrid_label.show()
                    self.hybrid_list.show()
                except Exception as e:
                    print(f"Error in hybrid recommendations: {e}")
                    self.hybrid_label.hide()
                    self.hybrid_list.hide()
            else:
                # Hide hybrid section if not logged in
                self.hybrid_label.hide()
                self.hybrid_list.hide()
    
    def hide_suggestions(self):
        self.suggestion_list.hide()


## Main Application Window

In [36]:
# Main Application Window
class MusicRecommendationApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.current_user = None
        self.initUI()
        
    def initUI(self):
        self.setWindowTitle("Music Discovery - AI Recommendation System")
        self.setGeometry(100, 100, 1400, 900)
        
        # Set application icon (you can add an icon file)
        # self.setWindowIcon(QIcon('icon.png'))
        
        # Central widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        
        # Stacked widget for different pages
        self.stacked_widget = QStackedWidget()
        
        # Create pages
        self.home_page = HomePage(self)
        self.login_page = LoginPage(self)
        self.filter_page = FilterPage(self)
        
        # Add pages to stacked widget
        self.stacked_widget.addWidget(self.home_page)
        self.stacked_widget.addWidget(self.login_page)
        self.stacked_widget.addWidget(self.filter_page)
        
        # Set main layout
        layout = QVBoxLayout()
        layout.setContentsMargins(0, 0, 0, 0)
        layout.addWidget(self.stacked_widget)
        central_widget.setLayout(layout)
        
        # Apply modern styling
        self.apply_styles()
    
    def apply_styles(self):
        self.setStyleSheet("""
            QMainWindow {
                background: qlineargradient(x1:0, y1:0, x2:0, y2:1,
                    stop:0 #f8f9fa, stop:1 #e9ecef);
            }
            
            QScrollBar:vertical {
                border: none;
                background: #f1f3f4;
                width: 8px;
                border-radius: 4px;
            }
            
            QScrollBar::handle:vertical {
                background: #c1c1c1;
                border-radius: 4px;
                min-height: 20px;
            }
            
            QScrollBar::handle:vertical:hover {
                background: #a8a8a8;
            }
            
            QScrollBar::add-line:vertical, QScrollBar::sub-line:vertical {
                height: 0px;
            }
        """)
    
    def update_recommendations(self):
        # Update user recommendations if logged in
        if self.current_user:
            try:
                # Clean up any existing song widgets
                for i in range(self.home_page.user_rec_list.count()):
                    item = self.home_page.user_rec_list.item(i)
                    widget = self.home_page.user_rec_list.itemWidget(item)
                    if widget and hasattr(widget, 'cleanup'):
                        try:
                            widget.cleanup()
                        except:
                            pass
                
                recommendations = recommend_by_user_id(self.current_user)
                self.home_page.user_rec_list.clear()
                
                if isinstance(recommendations, str):
                    # Error or user not found
                    error_item = QListWidgetItem()
                    error_widget = QLabel(recommendations)
                    error_widget.setStyleSheet("""
                        padding: 20px;
                        color: #856404;
                        font-size: 16px;
                        text-align: center;
                        background-color: #fff3cd;
                        border-radius: 12px;
                        margin: 8px;
                        border: 0px;
                    """)
                    error_widget.setAlignment(Qt.AlignCenter)
                    error_widget.setWordWrap(True)
                    error_item.setSizeHint(error_widget.sizeHint())
                    self.home_page.user_rec_list.addItem(error_item)
                    self.home_page.user_rec_list.setItemWidget(error_item, error_widget)
                else:
                    for _, row in recommendations.iterrows():
                        list_item = QListWidgetItem()
                        song_widget = SongWidget(
                            row['name'], 
                            row['artist'], 
                            row['spotify_preview_url'],
                            row['duration']
                        )
                        list_item.setSizeHint(song_widget.sizeHint())
                        self.home_page.user_rec_list.addItem(list_item)
                        self.home_page.user_rec_list.setItemWidget(list_item, song_widget)
            except Exception as e:
                print(f"Error updating recommendations: {e}")
                # Show error message
                error_item = QListWidgetItem()
                error_widget = QLabel("Unable to load recommendations at this time")
                error_widget.setStyleSheet("""
                    padding: 20px;
                    color: #dc3545;
                    font-size: 16px;
                    text-align: center;
                    border: 0px;
                """)
                error_widget.setAlignment(Qt.AlignCenter)
                error_item.setSizeHint(error_widget.sizeHint())
                self.home_page.user_rec_list.addItem(error_item)
                self.home_page.user_rec_list.setItemWidget(error_item, error_widget)
        else:
            # Show login prompt
            self.home_page.user_rec_list.clear()
            login_prompt = QWidget()
            login_prompt.setStyleSheet("""
                QWidget {
                    background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                        stop:0 #f8f9fa, stop:1 #e9ecef);
                    border-radius: 12px;
                    margin: 8px;
                }
            """)
            
            prompt_layout = QVBoxLayout()
            prompt_layout.setContentsMargins(20, 20, 20, 20)
            prompt_layout.setSpacing(12)
            
            prompt_title = QLabel("Get Personalized Recommendations")
            prompt_title.setStyleSheet("""
                font-size: 16px;
                font-weight: 600;
                color: #495057;
                text-align: center;
                border: 0px;
            """)
            prompt_title.setAlignment(Qt.AlignCenter)
            prompt_layout.addWidget(prompt_title)
            
            prompt_text = QLabel("Login to discover music based on your listening history and preferences.")
            prompt_text.setStyleSheet("""
                font-size: 14px;
                color: #6c757d;
                text-align: center;
                border: 0px;
            """)
            prompt_text.setAlignment(Qt.AlignCenter)
            prompt_text.setWordWrap(True)
            prompt_layout.addWidget(prompt_text)
            
            prompt_btn = QPushButton("Login Now")
            prompt_btn.setStyleSheet("""
                QPushButton {
                    background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                        stop:0 #007bff, stop:1 #0056b3);
                    color: white;
                    border: none;
                    padding: 12px 24px;
                    border-radius: 8px;
                    font-size: 14px;
                    font-weight: 600;
                }
                QPushButton:hover {
                    background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                        stop:0 #0056b3, stop:1 #004085);
                }
            """)
            prompt_btn.clicked.connect(lambda: self.stacked_widget.setCurrentIndex(1))
            prompt_layout.addWidget(prompt_btn, alignment=Qt.AlignCenter)
            
            login_prompt.setLayout(prompt_layout)
            
            list_item = QListWidgetItem()
            list_item.setSizeHint(login_prompt.sizeHint())
            self.home_page.user_rec_list.addItem(list_item)
            self.home_page.user_rec_list.setItemWidget(list_item, login_prompt)

## Run Application

In [37]:
def run():
    app = QApplication(sys.argv)
    
    # Set application properties
    app.setApplicationName("Music Discovery")
    app.setApplicationVersion("1.0")
    app.setOrganizationName("Music Recommendation System")
    
    # Create and show the main window
    window = MusicRecommendationApp()
    window.show()
    
    sys.exit(app.exec_())

run()

SystemExit: 0

c:\Users\tansiahong\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
